In [ ]:
def predict_signs(
    yolo: YOLO,
    resnets: Dict[int, nn.Module],
    image: np.ndarray,
    device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
) -> List[Dict]:
    """
    Two-stage detection on CPU/GPU.
    """
    
    img_h, img_w = image.shape[:2]
    
    # Normalization constants
    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    # Stage 1: YOLO detection
    results = yolo(image, conf=0.5, verbose=False)[0]
    
    if results.boxes is None:
        return []
    
    detections = []
    
    for box in results.boxes:
        # YOLO Ultralytics returns PIXEL coordinates
        x1_pixel, y1_pixel, x2_pixel, y2_pixel = map(int, box.xyxy[0].tolist())
        
        # Normalize to 0-1 range for comparison with ground truth
        x1 = x1_pixel / img_w
        y1 = y1_pixel / img_h
        x2 = x2_pixel / img_w
        y2 = y2_pixel / img_h
        
        # Теперь category напрямую соответствует GT категории!
        category = int(box.cls[0])
        yolo_conf = float(box.conf[0])
        
        # Extract crop using PIXEL coordinates
        crop = image[y1_pixel:y2_pixel, x1_pixel:x2_pixel]
        if crop.size == 0:
            continue
        
        # Get ResNet for this category (уже правильная!)
        resnet = resnets.get(category)
        if resnet is None:
            continue
        
        # Prepare crop for ResNet
        crop = cv2.resize(crop, (224, 224))
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        
        # Convert to tensor and normalize
        crop_tensor = torch.from_numpy(crop).permute(2, 0, 1).float() / 255.0
        crop_tensor = (crop_tensor - IMAGENET_MEAN) / IMAGENET_STD
        crop_tensor = crop_tensor.unsqueeze(0).to(device)
        
        # Stage 2: ResNet inference
        resnet = resnet.to(device)
        resnet.eval()
        
        with torch.no_grad():
            outputs = resnet(crop_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, local_class = torch.max(probs, dim=1)
        
        local_class = local_class.item()
        confidence = confidence.item()
        
        # Check if local_class exists in mapping
        if category not in LOCAL_TO_GLOBAL_MAPPING:
            continue
        if local_class not in LOCAL_TO_GLOBAL_MAPPING[category]:
            continue
        
        class_number, class_name = LOCAL_TO_GLOBAL_MAPPING[category][local_class]
        
        detections.append({
            "bbox": [x1, y1, x2, y2],
            "class_number": class_number,
            "class_name": class_name,
            "confidence": confidence,
            "yolo_confidence": yolo_conf
        })
    
    return detections

In [ ]:
def load_trained_resnets(resnet_dir: Path, category_signs: Dict[int, List[str]], device: torch.device) -> Dict[int, nn.Module]:
    """
    Load trained ResNet models for inference.
    
    Args:
        resnet_dir: Directory containing saved model weights
        category_signs: Dictionary mapping category_id to ordered list of sign names
        device: Device to load models on
    
    Returns:
        Dictionary of loaded ResNet models
    """
    resnets = {}
    
    for category_id in range(5):
        model_path = resnet_dir + f"/best_resnet_cat_{category_id}.pt"
        
        
        
        # Number of classes = number of signs in category + 1 (background)
        num_classes = len(category_signs.get(category_id, [])) + 1
        
        # Create model and load weights
        model = create_resnet_model(num_classes, freeze_backbone=False)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model = model.to(device)
        model.eval()
        
        resnets[category_id] = model
        print(f" Loaded ResNet for category {category_id} ({num_classes-1} sign classes)")
    
    return resnets


In [ ]:
def load_all_models(best_model_path = "../models/detect/cascade_model/weights/best.pt",
                   resnet_dir='../models/resnet_weights/prod',
                   device='cpu'):

    category_to_signs = {
        0: sorted([41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54]),
        1: sorted([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
        2: sorted([21, 22, 23, 24, 25, 26]),
        3: sorted([27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]),
        4: sorted([38, 39, 40])
    }

    
    yolo = YOLO(str(best_model_path)).to(device)
    resnets = load_trained_resnets(resnet_dir = resnet_dir, category_signs = category_to_signs, device=device)

    return yolo, resnets

In [ ]:
yolo, resnets = load_all_models()